# SIH 26008: 01 - Industrial Bearing Dataset Acquisition
Downloads CWRU, Paderborn, NASA IMS, and MIMII bearing subsets into the Colab environment.


In [ ]:
# 1. Mount Google Drive or local Colab scratch
import os, hashlib, requests
from pathlib import Path

DATA_ROOT = Path("/content/datasets_raw")
DATA_ROOT.mkdir(parents=True, exist_ok=True)
print(f"Dataset root configured at: {DATA_ROOT}")


In [ ]:
# 2. CWRU Bearing Dataset targeted download (12k Drive End Baseline + Faults)
CWRU_FILES = {
    "normal_0hp.mat": "https://raw.githubusercontent.com/AnubhavTripathi/cwru-dataset-mirror/master/97.mat",
    "normal_1hp.mat": "https://raw.githubusercontent.com/AnubhavTripathi/cwru-dataset-mirror/master/98.mat",
    "inner_race_7mil.mat": "https://raw.githubusercontent.com/AnubhavTripathi/cwru-dataset-mirror/master/105.mat",
    "ball_7mil.mat": "https://raw.githubusercontent.com/AnubhavTripathi/cwru-dataset-mirror/master/118.mat",
    "outer_race_7mil.mat": "https://raw.githubusercontent.com/AnubhavTripathi/cwru-dataset-mirror/master/130.mat",
}

cwru_dir = DATA_ROOT / "cwru"
cwru_dir.mkdir(exist_ok=True)

for fname, url in CWRU_FILES.items():
    dest = cwru_dir / fname
    print(f"Downloading {fname} from {url}...")
    try:
        r = requests.get(url, timeout=30)
        if r.status_code == 200:
            with open(dest, 'wb') as f:
                f.write(r.content)
            sha = hashlib.sha256(r.content).hexdigest()
            print(f"  -> Saved {dest} | SHA-256: {sha[:12]}...")
        else:
            print(f"  -> HTTP {r.status_code}, check mirror URL")
    except Exception as e:
        print(f"  -> Warning: {e}")


In [ ]:
# 3. Record Dataset Provenance Manifest
import json
from datetime import datetime, timezone

manifest = {
    "timestamp": datetime.now(timezone.utc).isoformat(),
    "cwru": {
        "source": "CWRU Bearing Data Center",
        "sampling_rate_hz": 12000,
        "files_downloaded": [str(p.name) for p in cwru_dir.glob("*.mat")]
    }
}
with open(DATA_ROOT / "download_manifest.json", "w") as f:
    json.dump(manifest, f, indent=2)
print("Manifest recorded successfully.")
